# Black-Litterman portfolio allocation for a net-zero transition

## What this notebook is trying to solve

The ETHack bonus question asks:

> If the world committed tomorrow to reaching net zero as fast as possible, how should a $1bn S&P 500 portfolio be allocated?

This notebook treats that as a portfolio-construction problem, not just a sustainability-ranking problem. The existing Climate Transition Score measures which companies look more prepared for a rapid emissions transition. The portfolio still needs to account for historical return risk, correlations, benchmark information and diversification constraints.

The core idea is therefore:

$$
\text{Net-zero allocation} = \text{market prior} + \text{transition-preparedness views} + \text{risk constraints}
$$

The final output is a table of optimized portfolio weights and dollar allocations for a $1bn long-only S&P 500 portfolio.


## Basic idea

A direct score-weighted portfolio would put the largest weights on the highest Climate Transition Scores. That is easy to understand, but it is not a complete investment model because it ignores three important facts:

1. High-scoring companies can still be risky or highly correlated with each other.
2. A low-scoring company can still belong in a diversified portfolio at a small weight.
3. The sustainability score is uncertain and should not be treated as a perfect return forecast.

Black-Litterman is useful because it starts with market-implied expected returns and then applies explicit, auditable views. In this notebook, the view is that companies with stronger transition preparedness should outperform companies with weaker preparedness if the world suddenly accelerates toward net zero.


## Black-Litterman intuition and notation

The Black-Litterman prior is the vector of equilibrium excess returns implied by the market portfolio:

$$
\Pi = \delta \Sigma w_{mkt}
$$

where:

- $\Sigma$ is the annualized covariance matrix of stock returns.
- $w_{mkt}$ is the benchmark market portfolio.
- $\delta$ is the investor's risk-aversion parameter.
- $\Pi$ is the prior expected-return vector implied by market equilibrium.

The model then combines $\Pi$ with investor views:

- $P$ maps each view to assets. A row of $P$ is a long-short portfolio.
- $Q$ is the expected return for each view.
- $\Omega$ is the uncertainty of each view.
- $\tau$ controls uncertainty in the prior.
- $\mu_{BL}$ is the posterior expected-return vector after blending the prior and the views.

In this notebook, the Climate Transition Score is interpreted as **transition preparedness**. It enters through $P$, $Q$ and $\Omega$, not as a direct portfolio weight. This preserves the distinction between a sustainability signal and an investable portfolio.


## Pipeline

The notebook follows the same transparent logic as the Climate Transition Score notebook:

1. Load the already available project data.
2. Align the common S&P 500 ticker universe.
3. Compute daily returns and an annualized covariance matrix.
4. Build the market-equilibrium prior $\Pi=\delta\Sigma w_{mkt}$.
5. Convert the Climate Transition Score into a relative Black-Litterman view.
6. Set view uncertainty $\Omega$ using data coverage as a confidence proxy.
7. Compute posterior expected returns $\mu_{BL}$.
8. Solve a constrained mean-variance portfolio.
9. Report weights, $1bn dollar allocations and diagnostics.


## Assumptions

The portfolio result depends on explicit assumptions. They are centralized in the parameter block below so the model can be tuned without hunting through the notebook.

The default configuration is intentionally conservative and presentation-friendly:

- **Prices:** local adjusted daily prices from `data/sp500_10yr_prices.csv`.
- **Sustainability:** local Climate Transition Score from `outputs/climate_transition_scores.csv`.
- **Returns:** daily simple returns.
- **Annualization:** 252 trading days.
- **Covariance:** annualized sample covariance of daily returns.
- **Benchmark prior:** market-cap weights if `data/sp500_market_caps.csv` exists; otherwise an equal-weight S&P 500 proxy.
- **Risk aversion:** $\delta=2.5$, a moderate standard choice for annual expected-return units.
- **Prior uncertainty:** $\tau=0.05$.
- **Transition view:** top-quintile transition-prepared firms outperform bottom-quintile firms by 3% per year.
- **View confidence:** average data coverage of the view companies, clipped between 25% and 85%.
- **Portfolio constraints:** long-only, fully invested, maximum 5% issuer weight.

The 3% view is not estimated from history. It is a scenario assumption: a rapid global net-zero commitment should create a relative return advantage for companies that are already better prepared for transition costs, regulation, customer shifts and stranded-asset risk.


## 1. Load libraries and define file paths

The notebook uses standard scientific Python tools. PyPortfolioOpt is used when available because it provides established portfolio routines. A short closed-form Black-Litterman and SciPy optimization fallback is kept so the notebook remains reproducible in a minimal environment.


In [29]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pypfopt import EfficientFrontier
    from pypfopt.black_litterman import BlackLittermanModel
except Exception:
    EfficientFrontier = None
    BlackLittermanModel = None

try:
    from scipy.optimize import minimize
except Exception:
    minimize = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

# Make the notebook robust whether it is run from the repository root or from notebooks/.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "sp500_10yr_prices.csv").exists()
)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print(f"Project root: {PROJECT_ROOT}")


Project root: c:\Users\axeli\Desktop\Coding\GitHub\Competitions\ETHack2026


## 2. Parameter block: tune the model here

This is the main control panel for the portfolio model. Change these values to study how the optimal allocation reacts to different assumptions.

The most important parameters are:

- `BL_VIEW_ANNUAL_OUTPERFORMANCE`: how strongly the net-zero scenario favors transition-prepared firms.
- `VIEW_TOP_QUANTILE` and `VIEW_BOTTOM_QUANTILE`: how broad or selective the sustainability view is.
- `VIEW_CONFIDENCE_MIN` and `VIEW_CONFIDENCE_MAX`: how much the model is allowed to trust the sustainability data.
- `RISK_AVERSION_DELTA`: how much the optimizer penalizes portfolio variance.
- `MAX_ASSET_WEIGHT`: how concentrated the portfolio is allowed to become.
- `TAU`: how uncertain the Black-Litterman prior is.

A softer sustainability adoption usually means lowering `BL_VIEW_ANNUAL_OUTPERFORMANCE`, widening the quantile groups, lowering `VIEW_CONFIDENCE_MAX`, or tightening `MAX_ASSET_WEIGHT`.


In [30]:
# -----------------------------------------------------------------------------
# DATA INPUTS
# -----------------------------------------------------------------------------
# The notebook uses files that already exist in the project. It deliberately does
# not download data, so results are reproducible from the repository contents.
PRICE_FILE = "sp500_10yr_prices.csv"
SCORE_FILE = "climate_transition_scores.csv"
MARKET_CAP_FILE = "sp500_market_caps.csv"  # Optional. If absent, use equal weights as the benchmark proxy.

# -----------------------------------------------------------------------------
# RETURN AND RISK MODEL
# -----------------------------------------------------------------------------
# Daily stock returns are annualized with 252 trading days, the standard equity
# convention. Change this only if the input data frequency changes.
TRADING_DAYS = 252

# Minimum fraction of price observations required before a stock is included.
# Higher values produce a cleaner but smaller universe. Lower values keep more
# stocks but may rely on more forward-filled prices.
MIN_PRICE_COVERAGE = 0.80
MIN_RETURN_COVERAGE = 0.95

# The baseline uses a sample covariance matrix because it is transparent and easy
# to explain. A shrinkage estimator would be a good extension for production use.
COVARIANCE_METHOD = "sample"

# -----------------------------------------------------------------------------
# BLACK-LITTERMAN PRIOR
# -----------------------------------------------------------------------------
# delta controls the trade-off between expected return and variance. Larger delta
# makes the optimizer more conservative; smaller delta allows more risk-taking.
RISK_AVERSION_DELTA = 2.5

# tau scales uncertainty in the market-equilibrium prior. Larger tau lets views
# move posterior returns more; smaller tau keeps the posterior closer to the prior.
TAU = 0.05

# -----------------------------------------------------------------------------
# SUSTAINABILITY VIEW
# -----------------------------------------------------------------------------
# The BL view is relative: high transition-preparedness stocks outperform low
# transition-preparedness stocks by this annual amount under rapid net zero.
# Lower this value to soften adoption of the sustainability ranking.
BL_VIEW_ANNUAL_OUTPERFORMANCE = 0.015

# Quantiles defining the long and short sides of the transition view. The default
# is top 20% versus bottom 20%. Use 0.67 / 0.33 for a broader, softer tercile view.
VIEW_TOP_QUANTILE = 0.70
VIEW_BOTTOM_QUANTILE = 0.30

# Confidence controls Omega. Higher confidence lowers Omega and makes the BL view
# stronger. The cap prevents the model from treating ESG data as certain.
VIEW_CONFIDENCE_MIN = 0.25
VIEW_CONFIDENCE_MAX = 0.85

# Multiplier on Omega. Use values above 1 to weaken the sustainability view without
# changing Q. For example, 2.0 doubles view uncertainty.
OMEGA_MULTIPLIER = 1.0

# -----------------------------------------------------------------------------
# PORTFOLIO CONSTRAINTS
# -----------------------------------------------------------------------------
# Total capital to allocate. The weights are unitless; this value only converts
# weights into dollar allocations.
PORTFOLIO_VALUE = 1_000_000_000

# Maximum individual issuer weight. Lower values force more diversification.
MAX_ASSET_WEIGHT = 0.05

# Numerical optimizer settings used by the SciPy fallback. PyPortfolioOpt ignores
# these, but they are useful when the fallback optimizer is used.
OPTIMIZER_MAX_ITER = 1_000
OPTIMIZER_FTOL = 1e-10

# -----------------------------------------------------------------------------
# OUTPUT FILES
# -----------------------------------------------------------------------------
ALLOCATION_OUTPUT_FILE = "black_litterman_net_zero_allocations.csv"
COMPARISON_OUTPUT_FILE = "black_litterman_vs_mean_variance_comparison.csv"

PRICE_PATH = DATA_DIR / PRICE_FILE
SCORE_PATH = OUTPUT_DIR / SCORE_FILE
CAP_PATH = DATA_DIR / MARKET_CAP_FILE
ALLOCATION_OUTPUT_PATH = OUTPUT_DIR / ALLOCATION_OUTPUT_FILE
COMPARISON_OUTPUT_PATH = OUTPUT_DIR / COMPARISON_OUTPUT_FILE

model_parameters = pd.Series({
    "TRADING_DAYS": TRADING_DAYS,
    "MIN_PRICE_COVERAGE": MIN_PRICE_COVERAGE,
    "MIN_RETURN_COVERAGE": MIN_RETURN_COVERAGE,
    "COVARIANCE_METHOD": COVARIANCE_METHOD,
    "RISK_AVERSION_DELTA": RISK_AVERSION_DELTA,
    "TAU": TAU,
    "BL_VIEW_ANNUAL_OUTPERFORMANCE": BL_VIEW_ANNUAL_OUTPERFORMANCE,
    "VIEW_TOP_QUANTILE": VIEW_TOP_QUANTILE,
    "VIEW_BOTTOM_QUANTILE": VIEW_BOTTOM_QUANTILE,
    "VIEW_CONFIDENCE_MIN": VIEW_CONFIDENCE_MIN,
    "VIEW_CONFIDENCE_MAX": VIEW_CONFIDENCE_MAX,
    "OMEGA_MULTIPLIER": OMEGA_MULTIPLIER,
    "PORTFOLIO_VALUE": PORTFOLIO_VALUE,
    "MAX_ASSET_WEIGHT": MAX_ASSET_WEIGHT,
})

model_parameters


TRADING_DAYS                            252
MIN_PRICE_COVERAGE                      0.8
MIN_RETURN_COVERAGE                    0.95
COVARIANCE_METHOD                    sample
RISK_AVERSION_DELTA                     2.5
TAU                                    0.05
BL_VIEW_ANNUAL_OUTPERFORMANCE         0.015
VIEW_TOP_QUANTILE                       0.7
VIEW_BOTTOM_QUANTILE                    0.3
VIEW_CONFIDENCE_MIN                    0.25
VIEW_CONFIDENCE_MAX                    0.85
OMEGA_MULTIPLIER                        1.0
PORTFOLIO_VALUE                  1000000000
MAX_ASSET_WEIGHT                       0.05
dtype: object

## 3. Load and inspect the data

This section loads the two portfolio inputs chosen in the parameter block and checks their dimensions before any modeling is done. The price file is already present in the project, so no external market-data download is required.


In [31]:
prices_raw = pd.read_csv(PRICE_PATH, parse_dates=["Date"])
scores_raw = pd.read_csv(SCORE_PATH)

data_inputs = pd.DataFrame([
    {"Dataset": "Adjusted stock prices", "Path": str(PRICE_PATH.relative_to(PROJECT_ROOT)), "Rows": len(prices_raw), "Columns": prices_raw.shape[1]},
    {"Dataset": "Climate Transition Scores", "Path": str(SCORE_PATH.relative_to(PROJECT_ROOT)), "Rows": len(scores_raw), "Columns": scores_raw.shape[1]},
])

data_inputs


,Dataset,Path,Rows,Columns
0,Adjusted stock prices,data\sp500_10yr_prices.csv,2512,504
1,Climate Transition Scores,outputs\climate_transition_scores.csv,500,29


## 4. Align the common ticker universe

The sustainability file uses Bloomberg-style identifiers such as `AAPL UW Equity`. The price file uses clean stock tickers such as `AAPL`. The join key is therefore the first token of the Bloomberg identifier.

The model keeps only companies with both a transition score and enough price history for return estimation. This prevents the optimizer from allocating to names whose risk cannot be estimated reliably.


In [32]:
prices = prices_raw.set_index("Date").sort_index()
prices.columns = prices.columns.astype(str).str.strip().str.replace("/", "-", regex=False)
prices = prices.apply(pd.to_numeric, errors="coerce")

scores = scores_raw.copy()
scores["Ticker"] = scores["ID"].astype(str).str.split().str[0].str.replace("/", "-", regex=False)
scores = (
    scores.dropna(subset=["Ticker", "Climate_Transition_Score"])
    .drop_duplicates("Ticker")
    .set_index("Ticker")
)

common_tickers = prices.columns.intersection(scores.index).sort_values()
prices = prices[common_tickers]
scores = scores.loc[common_tickers]

# Require broad price coverage, then forward-fill short gaps in adjusted prices.
prices = prices.dropna(axis=1, thresh=int(MIN_PRICE_COVERAGE * len(prices))).ffill().dropna()
scores = scores.loc[prices.columns]

universe_summary = pd.Series({
    "Common tickers before price-history filter": len(common_tickers),
    "Tickers after price-history filter": len(prices.columns),
    "First price date": prices.index.min().date(),
    "Last price date": prices.index.max().date(),
})

universe_summary


Common tickers before price-history filter           500
Tickers after price-history filter                   473
First price date                              2018-08-02
Last price date                               2026-09-11
dtype: object

## 5. Compute returns and covariance

The optimizer needs a risk model. This notebook uses daily simple returns and an annualized sample covariance matrix:

$$
\Sigma = 252 \times \operatorname{Cov}(r_{daily})
$$

A sample covariance estimator is transparent and easy to audit. More advanced estimators, such as Ledoit-Wolf shrinkage, would be reasonable extensions but are not necessary for the baseline notebook.


In [ ]:
returns = prices.pct_change().dropna(how="all")
returns = returns.dropna(axis=1, thresh=int(MIN_RETURN_COVERAGE * len(returns))).dropna()

if COVARIANCE_METHOD != "sample":
    raise ValueError("Only COVARIANCE_METHOD='sample' is implemented in this transparent baseline.")
Sigma = returns.cov() * TRADING_DAYS
scores = scores.loc[returns.columns]

risk_summary = pd.Series({
    "Return observations": len(returns),
    "Assets in covariance matrix": Sigma.shape[0],
    "Average annualized volatility": np.sqrt(np.diag(Sigma)).mean(),
    "Median annualized volatility": np.median(np.sqrt(np.diag(Sigma))),
})

risk_summary


## 6. Build the market-equilibrium prior

Black-Litterman starts from the return vector implied by the benchmark portfolio:

$$
\Pi = \delta \Sigma w_{mkt}
$$

A true S&P 500 prior should use market-cap weights. If a local market-cap file is not available, the notebook uses an equal-weight proxy and labels it clearly. This is preferable to inventing market capitalizations from price levels, because price alone does not identify company size.


In [ ]:
risk_aversion_delta = RISK_AVERSION_DELTA

default_weight = 1 / Sigma.shape[0]

if CAP_PATH.exists():
    caps = pd.read_csv(CAP_PATH)
    caps.columns = caps.columns.astype(str).str.strip()
    ticker_col = next(c for c in caps.columns if c.lower() in {"ticker", "symbol", "id"})
    cap_col = next(c for c in caps.columns if c.lower() in {"market_cap", "market cap", "mkt_cap", "capitalization"})
    caps[ticker_col] = caps[ticker_col].astype(str).str.split().str[0].str.replace("/", "-", regex=False)
    market_caps = caps.set_index(ticker_col)[cap_col].astype(float).reindex(Sigma.index)
    market_caps = market_caps.fillna(market_caps.dropna().mean())
    w_mkt = market_caps / market_caps.sum()
    benchmark_label = "market-cap weighted proxy"
else:
    w_mkt = pd.Series(default_weight, index=Sigma.index, name="Benchmark_Weight")
    benchmark_label = "equal-weight proxy"

pi = pd.Series(risk_aversion_delta * Sigma.values @ w_mkt.values, index=Sigma.index, name="Prior_Return")

prior_summary = pd.Series({
    "Benchmark used": benchmark_label,
    "Risk aversion delta": risk_aversion_delta,
    "Minimum benchmark weight": w_mkt.min(),
    "Maximum benchmark weight": w_mkt.max(),
    "Average prior return": pi.mean(),
})

prior_summary


## 7. Convert transition preparedness into BL views

The Climate Transition Score is strongest as a cross-sectional ranking. The parameterized baseline view is therefore relative rather than absolute:

$$
\text{Top transition-prepared quintile} - \text{Bottom transition-prepared quintile} = 3\%
$$

This says that, under a rapid net-zero scenario, companies already prepared for emissions reduction should earn a positive annual return premium over companies that look least prepared. The view is expressed as a long-short portfolio row in $P$: equal-weight long the top quintile and equal-weight short the bottom quintile.


In [ ]:
score = scores["Climate_Transition_Score"].astype(float)
high = score[score >= score.quantile(VIEW_TOP_QUANTILE)].index
low = score[score <= score.quantile(VIEW_BOTTOM_QUANTILE)].index

P = pd.DataFrame(0.0, index=["High minus low transition preparedness"], columns=Sigma.columns)
P.loc[:, high] = 1 / len(high)
P.loc[:, low] = -1 / len(low)

Q = pd.Series([BL_VIEW_ANNUAL_OUTPERFORMANCE], index=P.index, name="View_Return")

view_composition = pd.DataFrame({
    "Group": ["High transition preparedness", "Low transition preparedness"],
    "Stocks": [len(high), len(low)],
    "Average score": [score.loc[high].mean(), score.loc[low].mean()],
    "Average data coverage": [scores.loc[high, "Data_Coverage"].astype(float).mean(), scores.loc[low, "Data_Coverage"].astype(float).mean()],
})

view_composition


## 8. Define view uncertainty $\Omega$

The view uncertainty should be smaller when the view is more reliable and larger when the view is noisy. This notebook uses two ingredients:

1. The historical variance of the long-short view portfolio, $P\Sigma P^\top$.
2. The average data coverage of the companies used in the view.

The confidence value is clipped between 25% and 85%. The cap avoids treating ESG data as perfect even when coverage is high; the floor avoids discarding the view completely when coverage is imperfect.


In [ ]:
tau = TAU
view_variance = float((P.values @ Sigma.values @ P.values.T)[0, 0])
avg_coverage = scores.loc[high.union(low), "Data_Coverage"].astype(float).mean()
view_confidence = float(np.clip(avg_coverage, VIEW_CONFIDENCE_MIN, VIEW_CONFIDENCE_MAX))

Omega = pd.DataFrame(
    [[OMEGA_MULTIPLIER * view_variance * (1 - view_confidence) / view_confidence]],
    index=P.index,
    columns=P.index,
)

view_parameters = pd.DataFrame({
    "View": P.index,
    "Q annual return": Q.values,
    "Long-short variance": view_variance,
    "Confidence": view_confidence,
    "Omega": np.diag(Omega),
    "Tau": tau,
})

view_parameters


## 9. Compute Black-Litterman posterior returns

The posterior return vector $\mu_{BL}$ blends the market prior with the transition view. Companies in the high-score group should generally receive positive posterior revisions, while companies in the low-score group should generally receive negative revisions. The size of the revision depends on covariance and view confidence, not just the raw score.


In [ ]:
if BlackLittermanModel is not None:
    bl = BlackLittermanModel(
        Sigma,
        pi=pi,
        P=P,
        Q=Q,
        omega=Omega,
        tau=tau,
        risk_aversion=risk_aversion_delta,
    )
    mu_bl = bl.bl_returns().rename("BL_Posterior_Return")
else:
    # Standard BL posterior mean; all inputs are annualized to match Sigma and Q.
    tau_Sigma_inv = np.linalg.inv(tau * Sigma.values)
    Omega_inv = np.linalg.inv(Omega.values)
    posterior_precision = tau_Sigma_inv + P.values.T @ Omega_inv @ P.values
    posterior_mean_term = tau_Sigma_inv @ pi.values + P.values.T @ Omega_inv @ Q.values
    mu_bl = pd.Series(
        np.linalg.solve(posterior_precision, posterior_mean_term),
        index=Sigma.index,
        name="BL_Posterior_Return",
    )

posterior = pd.DataFrame({
    "Company": scores["Company"],
    "Sector": scores["GICS Sector"],
    "Climate_Transition_Score": score,
    "Prior_Return": pi,
    "BL_Posterior_Return": mu_bl,
    "Posterior_Minus_Prior": mu_bl - pi,
}).sort_values("Posterior_Minus_Prior", ascending=False)

posterior.head(10)


## 10. Optimize the constrained portfolio

The final allocation solves a long-only quadratic-utility problem:

$$
\max_w \; \mu_{BL}^{\top}w - \frac{\delta}{2} w^{\top}\Sigma w
$$

subject to:

$$
\sum_i w_i = 1, \quad 0 \leq w_i \leq 5\%
$$

The 5% issuer cap prevents the optimizer from turning one or two attractive posterior returns into an unrealistic concentrated fund. The result is a diversified transition-aware portfolio, not a pure list of climate leaders.


In [ ]:
portfolio_value = PORTFOLIO_VALUE
max_weight = MAX_ASSET_WEIGHT

def optimize_quadratic_utility(expected_returns, covariance, risk_aversion, max_asset_weight):
    """Long-only quadratic-utility optimizer used for BL and historical mean-variance portfolios."""
    if EfficientFrontier is not None:
        ef = EfficientFrontier(expected_returns, covariance, weight_bounds=(0, max_asset_weight))
        ef.max_quadratic_utility(risk_aversion=risk_aversion)
        return pd.Series(ef.clean_weights(), name="Weight").reindex(expected_returns.index).fillna(0.0)

    if minimize is None:
        raise ImportError("Install PyPortfolioOpt or scipy to run the constrained optimization.")
    x0 = np.repeat(1 / len(expected_returns), len(expected_returns))
    bounds = [(0, max_asset_weight)] * len(expected_returns)
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    result = minimize(
        lambda w: -(expected_returns.values @ w - 0.5 * risk_aversion * w @ covariance.values @ w),
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": OPTIMIZER_MAX_ITER, "ftol": OPTIMIZER_FTOL},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return pd.Series(result.x, index=expected_returns.index, name="Weight")

weights = optimize_quadratic_utility(mu_bl, Sigma, risk_aversion_delta, max_weight)
weights = weights / weights.sum()

allocations = (
    pd.DataFrame({
        "Company": scores["Company"],
        "Sector": scores["GICS Sector"],
        "Climate_Transition_Score": score,
        "Data_Coverage": scores["Data_Coverage"],
        "Benchmark_Weight": w_mkt,
        "Prior_Return": pi,
        "BL_Posterior_Return": mu_bl,
        "Weight": weights,
    })
    .assign(
        Active_Weight=lambda df: df["Weight"] - df["Benchmark_Weight"],
        Dollar_Allocation=lambda df: df["Weight"] * portfolio_value,
    )
    .sort_values("Weight", ascending=False)
)

allocations.head(25)


## 11. Build a historical mean-variance comparison portfolio

A plain mean-variance portfolio is a useful benchmark because it shows what happens when the optimizer uses historical average returns directly:

$$
\mu_{hist} = 252 \times \bar r_{daily}
$$

The risk model and constraints are kept identical to the Black-Litterman portfolio. The only difference is the expected-return input:

- **Historical mean-variance:** uses $\mu_{hist}$ and $\Sigma$.
- **Black-Litterman:** uses $\mu_{BL}$ and $\Sigma$.

This comparison is important because historical mean returns are noisy. A mean-variance optimizer can chase past winners aggressively, while Black-Litterman anchors the forecast to the market prior and applies only the documented transition-preparedness view.


In [ ]:
mu_hist = (returns.mean() * TRADING_DAYS).rename("Historical_Mean_Return")

mv_weights = optimize_quadratic_utility(mu_hist, Sigma, risk_aversion_delta, max_weight)
mv_weights = mv_weights / mv_weights.sum()

mv_allocations = (
    pd.DataFrame({
        "Company": scores["Company"],
        "Sector": scores["GICS Sector"],
        "Climate_Transition_Score": score,
        "Data_Coverage": scores["Data_Coverage"],
        "Historical_Mean_Return": mu_hist,
        "Weight": mv_weights,
    })
    .assign(Dollar_Allocation=lambda df: df["Weight"] * portfolio_value)
    .sort_values("Weight", ascending=False)
)

mv_allocations.head(25)


## 12. Compare Black-Litterman and mean-variance allocations

The main 0-to-1 distance measure is **Active Share**, introduced by Cremers and Petajisto as a holdings-based measure of how different a portfolio is from a benchmark. Here it is applied symmetrically to compare the Black-Litterman portfolio with the historical mean-variance portfolio:

$$
\text{Active Share}_{BL,MV} = \frac{1}{2}\sum_i |w_{BL,i} - w_{MV,i}|
$$

For two long-only, fully invested portfolios, Active Share lies between 0 and 1. A value near 0 means the two portfolios are almost identical. A value near 1 means they hold substantially different names or weights.

The comparison also reports expected return, volatility, Sharpe-like return-to-risk ratio, concentration, transition score and sector differences. This helps distinguish a genuine transition-aware BL tilt from a portfolio driven mainly by noisy historical returns.


In [ ]:
comparison_weights = pd.DataFrame({
    "Company": scores["Company"],
    "Sector": scores["GICS Sector"],
    "Climate_Transition_Score": score,
    "Benchmark_Weight": w_mkt,
    "BL_Weight": weights,
    "MV_Weight": mv_weights,
}).assign(
    BL_minus_MV=lambda df: df["BL_Weight"] - df["MV_Weight"],
    Abs_BL_minus_MV=lambda df: (df["BL_Weight"] - df["MV_Weight"]).abs(),
    BL_Dollar_Allocation=lambda df: df["BL_Weight"] * portfolio_value,
    MV_Dollar_Allocation=lambda df: df["MV_Weight"] * portfolio_value,
)

active_share_bl_mv = 0.5 * comparison_weights["Abs_BL_minus_MV"].sum()

def portfolio_statistics(label, portfolio_weights, expected_returns):
    annual_return = float(portfolio_weights @ expected_returns.reindex(portfolio_weights.index))
    annual_volatility = float(np.sqrt(portfolio_weights @ Sigma.loc[portfolio_weights.index, portfolio_weights.index] @ portfolio_weights))
    return pd.Series({
        "Expected return": annual_return,
        "Volatility": annual_volatility,
        "Return / volatility": annual_return / annual_volatility,
        "Largest position": portfolio_weights.max(),
        "Effective number of holdings": 1 / np.square(portfolio_weights).sum(),
        "Weighted transition score": np.average(score.reindex(portfolio_weights.index), weights=portfolio_weights),
    }, name=label)

portfolio_comparison = pd.concat([
    portfolio_statistics("Black-Litterman", weights, mu_bl),
    portfolio_statistics("Mean-variance", mv_weights, mu_hist),
    portfolio_statistics("Benchmark proxy", w_mkt, pi),
], axis=1).T

portfolio_comparison.loc["Black-Litterman vs mean-variance", "Active Share"] = active_share_bl_mv
portfolio_comparison


## 13. Sanity checks

These checks tie the numerical output back to the investment story. A reasonable result should be fully invested, long-only, within the issuer cap, tilted toward stronger transition preparedness, and still diversified across sectors.


In [ ]:
portfolio_checks = pd.DataFrame({
    "Black-Litterman": {
        "Total weight": weights.sum(),
        "Total dollar allocation": (weights * portfolio_value).sum(),
        "Largest issuer weight": weights.max(),
        "Number of non-zero positions": (weights > 1e-6).sum(),
        "Weighted transition score": np.average(score.reindex(weights.index), weights=weights),
    },
    "Mean-variance": {
        "Total weight": mv_weights.sum(),
        "Total dollar allocation": (mv_weights * portfolio_value).sum(),
        "Largest issuer weight": mv_weights.max(),
        "Number of non-zero positions": (mv_weights > 1e-6).sum(),
        "Weighted transition score": np.average(score.reindex(mv_weights.index), weights=mv_weights),
    },
    "Benchmark proxy": {
        "Total weight": w_mkt.sum(),
        "Total dollar allocation": (w_mkt * portfolio_value).sum(),
        "Largest issuer weight": w_mkt.max(),
        "Number of non-zero positions": (w_mkt > 1e-6).sum(),
        "Weighted transition score": np.average(score.reindex(w_mkt.index), weights=w_mkt),
    },
})

portfolio_checks


## 14. Diagnostics

The diagnostics are deliberately compact:

- prior versus posterior returns shows how much the BL view changed expected returns;
- score versus weight compares whether BL and historical MV tilt differently toward transition preparedness;
- sector exposure compares whether one method creates larger sector concentrations;
- largest active differences show which holdings drive the Active Share between the two portfolios.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(posterior["Prior_Return"], posterior["BL_Posterior_Return"], alpha=0.65)
ax.axline((0, 0), slope=1, color="black", linewidth=1, linestyle="--")
ax.set_title("Prior vs Black-Litterman posterior returns")
ax.set_xlabel("Market-implied prior return")
ax.set_ylabel("BL posterior return")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(comparison_weights["Climate_Transition_Score"], comparison_weights["BL_Weight"], alpha=0.60, label="Black-Litterman")
ax.scatter(comparison_weights["Climate_Transition_Score"], comparison_weights["MV_Weight"], alpha=0.45, label="Mean-variance")
ax.set_title("Transition preparedness vs portfolio weight")
ax.set_xlabel("Climate Transition Score")
ax.set_ylabel("Portfolio weight")
ax.legend()
plt.show()

sector_comparison = pd.DataFrame({
    "Black-Litterman": comparison_weights.groupby("Sector", dropna=False)["BL_Weight"].sum(),
    "Mean-variance": comparison_weights.groupby("Sector", dropna=False)["MV_Weight"].sum(),
}).sort_values("Black-Litterman", ascending=False)

sector_comparison.plot(kind="bar", figsize=(9, 4), title="Sector exposure: Black-Litterman vs mean-variance")
plt.ylabel("Portfolio weight")
plt.tight_layout()
plt.show()


In [ ]:
largest_position_differences = comparison_weights.sort_values("Abs_BL_minus_MV", ascending=False)[[
    "Company", "Sector", "Climate_Transition_Score", "BL_Weight", "MV_Weight",
    "BL_minus_MV", "BL_Dollar_Allocation", "MV_Dollar_Allocation"
]].head(25)

largest_position_differences


## 15. Export portfolio output

The Black-Litterman allocation and the BL-versus-MV comparison are saved separately. The comparison export is useful for presentations because it shows which holdings explain the difference between the transition-aware BL portfolio and the historical mean-variance portfolio.


In [ ]:
allocations.to_csv(ALLOCATION_OUTPUT_PATH)
comparison_weights.to_csv(COMPARISON_OUTPUT_PATH)

print(f"Saved BL allocation output: {ALLOCATION_OUTPUT_PATH}")
print(f"Saved BL vs MV comparison output: {COMPARISON_OUTPUT_PATH}")
print(f"Rows exported: {len(allocations)}")


## How to tune the allocation

Use the parameter block to run controlled scenarios. Good experiments are:

- **Softer sustainability adoption:** lower `BL_VIEW_ANNUAL_OUTPERFORMANCE`, lower `VIEW_CONFIDENCE_MAX`, or increase `OMEGA_MULTIPLIER`.
- **Broader sustainability adoption:** set `VIEW_TOP_QUANTILE = 0.67` and `VIEW_BOTTOM_QUANTILE = 0.33` so the view uses terciles instead of quintiles.
- **More diversified portfolio:** lower `MAX_ASSET_WEIGHT` from 5% to 2% or 3%.
- **More conservative risk posture:** increase `RISK_AVERSION_DELTA`.
- **More view-driven BL posterior:** increase `TAU` or decrease `OMEGA_MULTIPLIER`.

The most useful diagnostic is not only expected return. Look at Active Share versus the historical mean-variance portfolio, weighted transition score, largest positions, and sector exposure together. A portfolio is more convincing when the transition tilt is visible but not created by a few concentrated bets.


## Final answer and interpretation

The answer is not “buy only the greenest companies.” Under the modeled rapid net-zero scenario, the $1bn portfolio should tilt toward companies with high transition preparedness, but only after accounting for covariance, benchmark information and concentration limits.

The historical mean-variance comparison shows why Black-Litterman is useful. A mean-variance optimizer uses historical average returns directly, which can make the allocation sensitive to noisy realized winners and losers. Black-Litterman instead anchors expected returns to the market-equilibrium prior and then applies a transparent transition-preparedness view.

The key quantitative comparison is **Active Share** between the BL and MV portfolios:

$$
\frac{1}{2}\sum_i |w_{BL,i} - w_{MV,i}|
$$

This number lies between 0 and 1 for long-only fully invested portfolios. Low Active Share means the two methods produce similar holdings. High Active Share means the BL transition view materially changes the allocation relative to historical mean-variance optimization.

Main strengths:

- Uses the existing Climate Transition Score rather than creating a separate sustainability metric.
- Uses local historical stock prices already available in the project.
- Separates sustainability judgement from portfolio construction.
- Makes $P$, $Q$, $\Omega$, $\tau$, $\Pi$ and $\mu_{BL}$ explicit.
- Compares BL against a plain historical mean-variance portfolio using the same risk model and constraints.
- Uses Active Share as an academically recognized 0-to-1 holdings-difference measure.
- Produces investable weights and dollar allocations for a $1bn fund.

Main limitations:

- The 3% transition premium is a scenario assumption, not a historical estimate.
- Historical mean returns are noisy, so the mean-variance comparison should be interpreted as a benchmark rather than a preferred forecast.
- The equal-weight benchmark fallback is not a true S&P 500 cap-weighted market portfolio.
- Sample covariance can be noisy for large equity universes.
- The sustainability score measures corporate transition preparedness, not direct transition beta or stranded-asset exposure.
- Sector neutrality is not imposed; sector exposure should therefore be reviewed in the diagnostics.

Possible extensions include using true S&P 500 market-cap weights, sector-relative BL views, covariance shrinkage, tracking-error limits, transaction costs, turnover constraints and multiple transition scenarios with different values of $Q$ and $\Omega$.

Reference for Active Share: Cremers and Petajisto, “How Active Is Your Fund Manager? A New Measure That Predicts Performance,” *Review of Financial Studies*, 2009.
